# 01 - CNN 图像分类

## 学习目标
- 理解卷积神经网络
- 掌握 CNN 构建
- CIFAR-10 图像分类实战

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

## 1. CIFAR-10 数据集

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f'训练集: {x_train.shape}')
print(f'测试集: {x_test.shape}')

# 归一化
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 可视化
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 2. 构建 CNN

In [ ]:
model = keras.Sequential([
    # 第一卷积块
    layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # 第二卷积块
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # 分类头
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 3. 数据增强

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# 可视化增强效果
plt.figure(figsize=(10, 5))
image = x_train[0:1]
for i in range(9):
    augmented = data_augmentation(image, training=True)
    plt.subplot(3, 3, i+1)
    plt.imshow(augmented[0])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4. 训练

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=3)
]

history = model.fit(
    data_augmentation(x_train, training=True), y_train,
    validation_data=(x_test, y_test),
    epochs=30,
    batch_size=64,
    callbacks=callbacks
)

## 5. 评估

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'测试准确率: {test_acc:.4f}')

# 绘制训练曲线
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='训练')
plt.plot(history.history['val_accuracy'], label='验证')
plt.title('准确率')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='训练')
plt.plot(history.history['val_loss'], label='验证')
plt.title('损失')
plt.legend()
plt.tight_layout()
plt.show()

## 小结
- CNN 通过卷积提取局部特征
- BatchNorm 加速训练
- 数据增强提高泛化能力